# 03.6 Feature Sanity Checks

**Purpose**: Validate `master_races_clean.csv` feature calculations, with strict mathematical bounds and reasonable domain limits.

**Output**: A set of anomaly reports (counts + samples) for each check so we can quickly spot broken feature logic.


In [9]:

import pandas as pd
import numpy as np
import re
from pathlib import Path

# Paths
PROCESSED_DATA_DIR = Path("data/processed")
MASTER_PATH = PROCESSED_DATA_DIR / "master_races_clean.csv"

print(f"Loading: {MASTER_PATH}")
master = pd.read_csv(MASTER_PATH, low_memory=False)
print(f"Rows: {len(master):,}")
print(f"Columns: {len(master.columns):,}")

# Keep some common identifiers for reporting
ID_COLS = [c for c in ["year", "round", "name", "date", "raceId", "driverId", "constructorId", "code"] if c in master.columns]
print("ID columns:", ID_COLS)


# Check for duplicate driver rows per race
dups = master.duplicated(subset=['year','round','driverId'], keep=False)
print("Duplicate rows per (year, round, driverId):", dups.sum())

# Check max drivers per constructor per race
drivers_per_constructor = (
    master.groupby(['year','round','constructorId'])['driverId']
    .nunique()
    .reset_index(name='driver_count')
)
print("Max drivers per constructor per race:", drivers_per_constructor['driver_count'].max())


Loading: data\processed\master_races_clean.csv
Rows: 12,789
Columns: 65
ID columns: ['year', 'round', 'name', 'date', 'raceId', 'driverId', 'constructorId', 'code']
Duplicate rows per (year, round, driverId): 0
Max drivers per constructor per race: 2


In [10]:
def _numeric(series):
    return pd.to_numeric(series, errors="coerce")


def _report_issues(title, mask, cols=None, sample_n=10):
    count = int(mask.sum()) if hasattr(mask, "sum") else int(np.sum(mask))
    total = len(master)
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)
    print(f"Issues: {count} / {total} ({(count / total * 100):.2f}%)")
    if count == 0:
        return

    cols = cols or ID_COLS
    show_cols = [c for c in cols if c in master.columns]
    extra_cols = [c for c in (cols or []) if c in master.columns and c not in show_cols]
    use_cols = show_cols + extra_cols
    print("Sample rows:")
    print(master.loc[mask, use_cols].head(sample_n))


def check_range(col, min_val=None, max_val=None, allow_na=True, title=None):
    if col not in master.columns:
        print(f"SKIP (missing column): {col}")
        return
    s = _numeric(master[col])
    mask = pd.Series(False, index=master.index)
    if min_val is not None:
        mask |= s < min_val
    if max_val is not None:
        mask |= s > max_val
    if allow_na:
        mask &= s.notna()
    title = title or f"Range check: {col} in [{min_val}, {max_val}]"
    _report_issues(title, mask, cols=ID_COLS + [col])


def check_rate(col, title=None):
    check_range(col, min_val=0, max_val=1, allow_na=True, title=title or f"Rate check: {col} in [0, 1]")


def check_int_nonneg(col, max_val=None, title=None):
    if col not in master.columns:
        print(f"SKIP (missing column): {col}")
        return
    s = _numeric(master[col])
    mask = (s < 0) & s.notna()
    if max_val is not None:
        mask |= (s > max_val) & s.notna()
    title = title or f"Non-negative integer check: {col}"
    _report_issues(title, mask, cols=ID_COLS + [col])


def check_relation(col_a, col_b, op, title):
    if col_a not in master.columns or col_b not in master.columns:
        print(f"SKIP (missing column): {col_a} or {col_b}")
        return
    a = _numeric(master[col_a])
    b = _numeric(master[col_b])
    mask = op(a, b) & a.notna() & b.notna()
    _report_issues(title, mask, cols=ID_COLS + [col_a, col_b])


def check_time_string(col, max_minutes=10, title=None):
    if col not in master.columns:
        print(f"SKIP (missing column): {col}")
        return

    def parse_time_to_td(val):
        if pd.isna(val):
            return pd.NaT
        s = str(val).strip()
        if s in ["", "\\N", "NaT", "None", "nan"]:
            return pd.NaT

        # Normalize formats like M:SS.mmm or H:MM:SS.mmm
        if re.match(r"^\d+:\d{2}\.\d+$", s):
            s = "00:" + s
        elif re.match(r"^\d+:\d{2}:\d{2}$", s):
            s = s + ".000"
        elif re.match(r"^\d+:\d{2}:\d{2}\.\d+$", s):
            pass
        else:
            return pd.NaT

        try:
            return pd.to_timedelta(s)
        except Exception:
            return pd.NaT

    td = master[col].apply(parse_time_to_td)
    mask = td.notna() & (td > pd.Timedelta(minutes=max_minutes))
    title = title or f"Time check: {col} <= {max_minutes} minutes"
    _report_issues(title, mask, cols=ID_COLS + [col])


def check_outliers_iqr(col, factor=3.0, title=None):
    if col not in master.columns:
        print(f"SKIP (missing column): {col}")
        return
    s = _numeric(master[col]).dropna()
    if len(s) < 10:
        print(f"SKIP (too few values): {col}")
        return
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo = q1 - factor * iqr
    hi = q3 + factor * iqr
    mask = _numeric(master[col]).between(lo, hi) == False
    mask &= _numeric(master[col]).notna()
    title = title or f"Outlier check (IQR {factor}x): {col} outside [{lo:.3f}, {hi:.3f}]"
    _report_issues(title, mask, cols=ID_COLS + [col])


In [11]:

# -----------------------------
# Core integrity checks
# -----------------------------
check_int_nonneg("resultId")
check_int_nonneg("raceId")
check_int_nonneg("driverId")
check_int_nonneg("constructorId")
check_int_nonneg("circuitId")

# Core racing fields (reasonable ranges)
check_range("grid", min_val=0, max_val=30, title="Grid in [0, 30]")
check_range("position", min_val=1, max_val=30, title="Position in [1, 30]")
check_range("positionOrder", min_val=1, max_val=30, title="PositionOrder in [1, 30]")
check_range("points", min_val=0, max_val=26, title="Points in [0, 26]")
check_range("laps", min_val=0, max_val=100, title="Laps in [0, 100]")

# Time sanity
check_time_string("q1", max_minutes=10, title="Q1 time <= 10 minutes")
check_time_string("q2", max_minutes=10, title="Q2 time <= 10 minutes")
check_time_string("q3", max_minutes=10, title="Q3 time <= 10 minutes")

check_range("milliseconds", min_val=0, max_val=6 * 60 * 60 * 1000, title="Milliseconds <= 6 hours")

# -----------------------------
# Feature sanity checks
# -----------------------------
# Rates must be in [0,1]
rate_cols = [
    "finished_lapped_rate_last_10",
    "dnf_rate_last_10",
    "disqualified_rate_last_10",
    "not_classified_rate_last_10",
    "driver_podium_rate_last_10",
    "driver_podium_rate_at_circuit",
    "constructor_podium_rate_at_circuit",
    "constructor_podium_rate_last_15",
]
for col in rate_cols:
    check_rate(col)

# Rolling averages (reasonable bounds)
check_range("driver_avg_position_last_5", min_val=1, max_val=30, title="driver_avg_position_last_5 in [1, 30]")
check_range("driver_avg_grid_last_5", min_val=1, max_val=30, title="driver_avg_grid_last_5 in [1, 30]")

# Pre-race standings
check_range("driver_standings_position_PRE_RACE", min_val=1, max_val=30, title="driver_standings_position_PRE_RACE in [1, 30]")
check_range("constructor_standings_position_PRE_RACE", min_val=1, max_val=20, title="constructor_standings_position_PRE_RACE in [1, 20]")
check_range("driver_standings_position", min_val=1, max_val=30, title="driver_standings_position in [1, 30]")
check_range("constructor_standings_position", min_val=1, max_val=20, title="constructor_standings_position in [1, 20]")

# Explicit PRE_RACE points checks (new)
check_int_nonneg("driver_standings_points_PRE_RACE", title="driver_standings_points_PRE_RACE non-negative")
check_int_nonneg("constructor_standings_points_PRE_RACE", title="constructor_standings_points_PRE_RACE non-negative")

# -----------------------------
# Standings points investigation
# -----------------------------
def standings_points_summary(df, points_col, group_col, label):
    if points_col not in df.columns or group_col not in df.columns:
        print(f"⚠ Missing columns for {label}: {points_col} or {group_col}")
        return

    s = pd.to_numeric(df[points_col], errors='coerce')

    print("\n" + "=" * 80)
    print(f"{label} standings points summary: {points_col}")
    print("=" * 80)
    print(f"Non-null: {s.notna().sum():,} / {len(s):,}")
    print(f"Min/Median/Max: {s.min()} / {s.median()} / {s.max()}")
    print(f"Negative values: {(s < 0).sum():,}")

    # By year: missing + max
    by_year = (
        df.assign(_pts=s)
          .groupby('year')['_pts']
          .agg(missing=lambda x: x.isna().sum(),
               min='min',
               median='median',
               max='max')
          .reset_index()
    )
    print("\nBy year (missing/min/median/max):")
    print(by_year.tail(10))

    # Non-decreasing check within each season
    tmp = df[['year', group_col, points_col, 'round']].copy()
    tmp[points_col] = pd.to_numeric(tmp[points_col], errors='coerce')
    tmp = tmp.sort_values(['year', 'round'])

    def count_decreases(grp):
        vals = grp[points_col].dropna()
        if len(vals) < 2:
            return 0
        return (vals.diff() < 0).sum()

    dec = tmp.groupby(['year', group_col]).apply(count_decreases)
    total_decreases = int(dec.sum())
    print(f"\nDecreases within season (should be 0 if cumulative): {total_decreases:,}")
    if total_decreases > 0:
        print("Sample groups with decreases:")
        print(dec[dec > 0].head(10))

# Driver standings points
standings_points_summary(master, 'driver_standings_points', 'driverId', 'Driver')

# Constructor standings points
standings_points_summary(master, 'constructor_standings_points', 'constructorId', 'Constructor')

# Driver PRE_RACE standings points (new)
standings_points_summary(master, 'driver_standings_points_PRE_RACE', 'driverId', 'Driver PRE_RACE')

# Constructor PRE_RACE standings points (new)
standings_points_summary(master, 'constructor_standings_points_PRE_RACE', 'constructorId', 'Constructor PRE_RACE')

# Investigate constructor standings decreases with context + all drivers
if 'constructor_standings_points' in master.columns:
    tmp = master[['year', 'round', 'date', 'raceId', 'constructorId', 'driverId', 'code',
                  'points', 'constructor_standings_points']].copy()
    tmp['constructor_standings_points'] = pd.to_numeric(tmp['constructor_standings_points'], errors='coerce')
    tmp = tmp.sort_values(['year', 'round', 'date'])

    # Find drops at constructor level
    tmp['prev_points'] = tmp.groupby(['year', 'constructorId'])['constructor_standings_points'].shift(1)
    drops = tmp[tmp['constructor_standings_points'] < tmp['prev_points']]

    print("\nConstructor standings decreases (with context):")
    for _, row in drops.drop_duplicates(subset=['year','constructorId','raceId']).iterrows():
        y = row['year']
        cid = row['constructorId']
        race_id = row['raceId']

        # previous + current + next race for that constructor (all drivers)
        cons_rows = tmp[(tmp['year'] == y) & (tmp['constructorId'] == cid)]
        cons_rows = cons_rows.sort_values(['round', 'date'])

        # identify previous and next raceId for this constructor
        race_ids = cons_rows['raceId'].dropna().unique().tolist()
        if race_id in race_ids:
            idx = race_ids.index(race_id)
            prev_race = race_ids[idx-1] if idx-1 >= 0 else None
            next_race = race_ids[idx+1] if idx+1 < len(race_ids) else None
        else:
            prev_race = next_race = None

        context_races = [r for r in [prev_race, race_id, next_race] if r is not None]

        print(f"\nYear {y} | constructorId {cid} | raceId {race_id}")
        print(cons_rows[cons_rows['raceId'].isin(context_races)]
              .sort_values(['raceId','driverId'])
              [['year','round','date','raceId','constructorId','driverId','code','points','constructor_standings_points']])

# Age and cumulative stats
check_range("driver_age", min_val=15, max_val=60, title="driver_age in [15, 60]")
check_int_nonneg("driver_total_podiums", max_val=400, title="driver_total_podiums (<= 400)")
check_int_nonneg("driver_races_completed", max_val=1000, title="driver_races_completed (<= 1000)")
check_int_nonneg("driver_races_at_circuit", max_val=100, title="driver_races_at_circuit (<= 100)")

# Relationship checks
check_relation(
    "driver_total_podiums",
    "driver_races_completed",
    lambda a, b: a > b,
    "driver_total_podiums should not exceed driver_races_completed",
)
check_relation(
    "driver_races_at_circuit",
    "driver_races_completed",
    lambda a, b: a > b,
    "driver_races_at_circuit should not exceed driver_races_completed",
)

# Trend sanity (conservative bounds)
check_range("positionOrder_trend_season", min_val=-20, max_val=20, title="positionOrder_trend_season in [-20, 20]")

# -----------------------------
# Outlier scans for key numeric features (IQR-based)
# -----------------------------
outlier_cols = [
    "driver_points_avg_last_10",
    "driver_avg_position_last_5",
    "driver_avg_grid_last_5",
    "driver_total_podiums",
    "driver_races_completed",
]
for col in outlier_cols:
    check_outliers_iqr(col, factor=3.0)

print("\nAll checks complete.")



Non-negative integer check: resultId
Issues: 0 / 12789 (0.00%)

Non-negative integer check: raceId
Issues: 0 / 12789 (0.00%)

Non-negative integer check: driverId
Issues: 0 / 12789 (0.00%)

Non-negative integer check: constructorId
Issues: 0 / 12789 (0.00%)

Non-negative integer check: circuitId
Issues: 0 / 12789 (0.00%)

Grid in [0, 30]
Issues: 0 / 12789 (0.00%)

Position in [1, 30]
Issues: 0 / 12789 (0.00%)

PositionOrder in [1, 30]
Issues: 0 / 12789 (0.00%)

Points in [0, 26]
Issues: 3 / 12789 (0.02%)
Sample rows:
       year  round                  name        date  raceId  driverId  \
10257  2014     19  Abu Dhabi Grand Prix  2014-11-23     918         1   
10258  2014     19  Abu Dhabi Grand Prix  2014-11-23     918        13   
10259  2014     19  Abu Dhabi Grand Prix  2014-11-23     918       822   

       constructorId code  points  
10257            131  HAM    50.0  
10258              3  MAS    36.0  
10259              3  BOT    30.0  

Laps in [0, 100]
Issues: 0 / 12789

C:\Users\erikv\AppData\Local\Temp\ipykernel_740\226712249.py:92: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  dec = tmp.groupby(['year', group_col]).apply(count_decreases)
C:\Users\erikv\AppData\Local\Temp\ipykernel_740\226712249.py:92: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  dec = tmp.groupby(['year', group_col]).apply(count_decreases)


In [12]:
# Show 2023 season for constructorId 9 (all rounds, all drivers)
cols = [c for c in [
    'year','round','date','raceId','name',
    'constructorId','driverId','code',
    'points','driver_standings_points','constructor_standings_points'
] if c in master.columns]

view = (
    master[(master['year'] == 2023) & (master['constructorId'] == 9)]
    .sort_values(['round','date','driverId'])
    [cols]
)

print(view)

       year  round        date  raceId                      name  \
1410   2023      1  2023-03-05    1098        Bahrain Grand Prix   
1409   2023      1  2023-03-05    1098        Bahrain Grand Prix   
12509  2023      2  2023-03-19    1099  Saudi Arabian Grand Prix   
12510  2023      2  2023-03-19    1099  Saudi Arabian Grand Prix   
542    2023      3  2023-04-02    1100     Australian Grand Prix   
538    2023      3  2023-04-02    1100     Australian Grand Prix   
12349  2023      4  2023-04-30    1101     Azerbaijan Grand Prix   
12350  2023      4  2023-04-30    1101     Azerbaijan Grand Prix   
12670  2023      5  2023-05-07    1102          Miami Grand Prix   
12669  2023      5  2023-05-07    1102          Miami Grand Prix   
2950   2023      6  2023-05-28    1104         Monaco Grand Prix   
2935   2023      6  2023-05-28    1104         Monaco Grand Prix   
2100   2023      7  2023-06-04    1105        Spanish Grand Prix   
2097   2023      7  2023-06-04    1105        Sp